# 🃏 Ace's — TensorFlow Training Server

**Run this notebook on Google Colab (GPU runtime recommended).**

### What it does
1. Installs dependencies (TensorFlow, Flask, ngrok)
2. Clones the `latest-ai` branch of the AIAces repo
3. Starts the training server with a public ngrok URL
4. You paste the ngrok URL into the Java game at startup
5. Training runs 24/7 — weights auto-pushed to git every 100 steps

### Setup
- Set `GITHUB_TOKEN` below (needs `repo` scope for pushing weights)
- Set `NGROK_TOKEN` below (free account at https://ngrok.com)
- Run all cells in order ▶▶▶

In [ ]:
# ════════════════════════════════════════════════════════════
# 🔧 CONFIGURATION — fill these in before running
# ════════════════════════════════════════════════════════════

GITHUB_TOKEN = ""   # GitHub personal access token (repo scope)
NGROK_TOKEN  = ""   # ngrok authtoken (https://dashboard.ngrok.com)

REPO_URL  = "https://github.com/simbaking/AIAces"
BRANCH    = "latest-ai"
PORT      = 5001

# Git identity used for auto-commits from Colab
GIT_EMAIL = "ai-trainer@aces.com"
GIT_NAME  = "Aces Trainer"

print("✅ Configuration loaded.")

In [ ]:
# ════════════════════════════════════════════════════════════
# 📦 Install dependencies
# ════════════════════════════════════════════════════════════
import os
import sys

print("📦 Installing Python dependencies...")
!pip install flask flask-cors pyngrok --quiet

print("📦 Installing maven (for headless Java client)...")
!apt-get update -qq && apt-get install -y maven --quiet

print("✅ Dependencies installed.")

In [ ]:
# ════════════════════════════════════════════════════════════
# 📥 Clone the repo (or pull latest if already cloned)
# --depth 1 = shallow clone: only the latest commit is fetched,
# skipping the full weight history (~3 GB saved on every clone).
# ════════════════════════════════════════════════════════════
import os
import subprocess

CLONE_DIR = "/content/aces-game"

# Build authenticated URL so weights can be pushed back
if GITHUB_TOKEN:
    auth_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
else:
    auth_url = REPO_URL
    print("⚠️  No GITHUB_TOKEN — auto git-push of weights will be disabled.")

if not os.path.exists(CLONE_DIR):
    print(f"📥 Shallow-cloning {BRANCH} branch (latest commit only)...")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "-b", BRANCH, auth_url, CLONE_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("❌ Clone failed:", result.stderr)
        raise RuntimeError("git clone failed — check GITHUB_TOKEN and repo URL")
    print("✅ Shallow clone complete.")
else:
    print("♻️  Repo already cloned — pulling latest...")
    !git -C {CLONE_DIR} remote set-url origin {auth_url}
    !git -C {CLONE_DIR} fetch --depth 1 origin {BRANCH}
    !git -C {CLONE_DIR} reset --hard origin/{BRANCH}

# Set git identity for auto-commits
!git -C {CLONE_DIR} config user.email "{GIT_EMAIL}"
!git -C {CLONE_DIR} config user.name  "{GIT_NAME}"

# Verify
!git -C {CLONE_DIR} log --oneline -3
print("✅ Repo ready.")

In [ ]:
# ════════════════════════════════════════════════════════════
# 🔄 Sync — always force-reset repo to latest remote code
#    Runs automatically. Discards any stale local commits
#    so you always get the exact code from latest-ai.
# ════════════════════════════════════════════════════════════
import subprocess

CLONE_DIR = "/content/aces-game"

print("🔄 Syncing repo to latest-ai...")

# Fetch latest from remote
r = subprocess.run(["git", "-C", CLONE_DIR, "fetch", "origin"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print("⚠️  Fetch failed:", r.stderr.strip())
else:
    print("  ✅ Fetched latest commits")

# Hard-reset working tree to exactly match remote
# (discards any stale local auto-weight commits that blocked pulls)
r = subprocess.run(["git", "-C", CLONE_DIR, "reset", "--hard", "origin/latest-ai"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print("⚠️  Reset failed:", r.stderr.strip())
else:
    print(" ", r.stdout.strip())

# Verify key endpoints exist in server.py
r = subprocess.run(["grep", "-c", "git_status",
                    f"{CLONE_DIR}/training_server/server.py"],
                   capture_output=True, text=True)
hits = int(r.stdout.strip()) if r.stdout.strip().isdigit() else 0
if hits > 0:
    print("  ✅ server.py is up-to-date")
else:
    print("  ⚠️  server.py looks outdated — check your GitHub token and repo URL")

# Show the 3 latest commits so you can confirm what version you have
r = subprocess.run(["git", "-C", CLONE_DIR, "log", "--oneline", "-3"],
                   capture_output=True, text=True)
print("\n  Latest commits on disk:")
for line in r.stdout.strip().splitlines():
    print("   ", line)
print("")

In [ ]:
# ════════════════════════════════════════════════════════════
# 🚀 Start the TF Training Server + ngrok tunnel
# ════════════════════════════════════════════════════════════
import os
import sys
import threading
import time
import subprocess
from pyngrok import ngrok, conf

CLONE_DIR   = "/content/aces-game"
SERVER_PATH = os.path.join(CLONE_DIR, "training_server", "server.py")

# Configure ngrok
if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN
else:
    print("⚠️  No NGROK_TOKEN — ngrok may rate-limit or fail.")

# Set environment variables for server.py
os.environ["REPO_ROOT"] = CLONE_DIR
os.environ["PORT"]      = str(PORT)
os.environ["HOST"]      = "0.0.0.0"
os.environ["GIT_PUSH"]  = "true" if GITHUB_TOKEN else "false"
os.environ["GITHUB_TOKEN"]  = GITHUB_TOKEN
os.environ["GITHUB_USER"]   = "simbaking"

# Add training_server dir to Python path
server_dir = os.path.join(CLONE_DIR, "training_server")
if server_dir not in sys.path:
    sys.path.insert(0, server_dir)

# Open ngrok tunnel BEFORE starting Flask (catch exceptions so Flask still starts)
public_url = None
try:
    print(f"🌐 Opening ngrok tunnel on port {PORT}...")
    tunnel = ngrok.connect(PORT, "http")
    public_url = tunnel.public_url
    print("\n" + "═" * 60)
    print(f"  🔗 Public URL: {public_url}")
    print("═" * 60 + "\n")
    print("👉 Paste this URL into the Java game when prompted at startup.")
    print(f"   e.g.  export TRAINING_SERVER_URL={public_url}")
    print("        then:  mvn spring-boot:run\n")
except Exception as e:
    print("\n❌  ngrok connection failed!\n")
    print(f"Error details: {e}\n")
    print("⚠️  IMPORTANT: If ngrok fails, check if your token is valid.")
    print("Starting the training server on localhost anyway...\n")

# Run server.py as a subprocess and stream its output to Colab cell output
def run_server():
    while True:
        print("🚀 Starting server subprocess...")
        try:
            process = subprocess.Popen(
                [sys.executable, SERVER_PATH],
                env=os.environ,
                cwd=server_dir,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1
            )
            
            # Stream output line by line as it is produced
            for line in iter(process.stdout.readline, ''):
                print(f"[Server] {line.rstrip()}", flush=True)
                
            process.wait()
            if process.returncode == 3:
                print("🔄 [Auto-Update] Reloading server subprocess due to code updates...\n")
                time.sleep(1)
                continue
            else:
                print(f"🛑 Server subprocess exited with code {process.returncode}. Restarting in 2s...")
        except Exception as e:
            print(f"❌ Failed to launch or monitor server subprocess: {e}")
        time.sleep(2)

server_thread = threading.Thread(target=run_server, daemon=True, name="TFServer")
server_thread.start()

# Wait for Flask to start (model build takes 30-90s on CPU)
print("⏳ Waiting for server to start (model building on CPU — up to 90s)...")
import urllib.request as _ur
_ready = False
for _i in range(90):
    time.sleep(1)
    try:
        _ur.urlopen(f"http://localhost:{PORT}/status", timeout=2)
        _ready = True
        print(f"✅ Server ready after {{_i+1}}s")
        break
    except:
        if (_i+1) % 10 == 0:
            print(f"  still waiting... {{_i+1}}s")
if not _ready:
    print("⚠️  Server did not respond after 90s — check the thread output above for errors")
status_url = public_url if public_url else f"http://localhost:{PORT}"
print("📊 Status:", status_url + "/status")

In [ ]:
# ════════════════════════════════════════════════════════════
# 📊 Check server status (run anytime)
# ════════════════════════════════════════════════════════════
import urllib.request, json

try:
    with urllib.request.urlopen(f"http://localhost:{PORT}/status", timeout=5) as r:
        status = json.loads(r.read())
    print("Server Status:")
    for k, v in status.items():
        print(f"  {k:25s}: {v}")
except Exception as e:
    print(f"Could not reach server: {e}")

In [ ]:
# ════════════════════════════════════════════════════════════
# 💾 Force push weights to git RIGHT NOW
# ════════════════════════════════════════════════════════════
import urllib.request, json

req = urllib.request.Request(
    f"http://localhost:{PORT}/push",
    data=b"", method="POST"
)
try:
    with urllib.request.urlopen(req, timeout=60) as r:
        result = json.loads(r.read())
except urllib.error.HTTPError as e:
    result = json.loads(e.read())

print(f"ok: {result.get('ok')}")
for line in result.get('log', []):
    print(line)
if 'error' in result:
    print(f"❌ {result['error']}")

In [ ]:
# ════════════════════════════════════════════════════════════
# 🔍 Git / Push Diagnostics — run this if push seems broken
# ════════════════════════════════════════════════════════════
import urllib.request, json

# 1️⃣  Check git state on the server
with urllib.request.urlopen(f"http://localhost:{PORT}/git_status", timeout=5) as r:
    gs = json.loads(r.read())
print("\n── Git Status ──────────────────────────")
for k, v in gs.items():
    print(f"  {k:15s}: {v}")

# 2️⃣  Force a synchronous push and show the full output
print("\n── Force Push ──────────────────────────")
req = urllib.request.Request(f"http://localhost:{PORT}/push", data=b"", method="POST")
try:
    with urllib.request.urlopen(req, timeout=60) as r:
        result = json.loads(r.read())
except urllib.error.HTTPError as e:
    result = json.loads(e.read())
print(f"  ok: {result.get('ok')}")
for line in result.get('log', []):
    print(f"  {line}")
if 'error' in result:
    print(f"  ❌ ERROR: {result['error']}")

In [ ]:
# ════════════════════════════════════════════════════════════
# ⏰ Keep-alive loop — prevents Colab from disconnecting
# Prints a status line every 10 minutes. Stop with ■.
# ════════════════════════════════════════════════════════════
import urllib.request, json, time

CHECK_INTERVAL = 600  # seconds

print("⏰ Keep-alive started. Press ■ to stop.")
while True:
    try:
        with urllib.request.urlopen(f"http://localhost:{PORT}/status", timeout=5) as r:
            s = json.loads(r.read())
        print(
            f"[{time.strftime('%H:%M:%S')}] "
            f"Steps={s.get('train_steps', 0):,}  "
            f"Samples={s.get('samples_received', 0):,}  "
            f"Buffer={s.get('buffer_size', 0):,}  "
            f"LastPush={s.get('last_git_push', 'never')}"
        )
    except Exception as e:
        print(f"[{time.strftime('%H:%M:%S')}] ⚠️  Server ping failed: {e}")
    time.sleep(CHECK_INTERVAL)